# 홈캠 이벤트 감지 AI 조합 비교·추천

- 대상: Jetson Orin Nano + Raspberry Pi 5
- 기준일: 2026-07-21
- 기존 확정 모델: YOLOv8n 사람 탐지, YAMNet TFLite 웃음 감지
- 현재 전제: 웹캠 영상과 녹화 파일은 Jetson이 소유하며, 마이크도 녹화 음성에 포함한다.

## 최종 선택 한눈에 보기

가장 좋은 구성은 **싼 신호처리 → 범용 AI 확인 → 여러 신호 결합 → 녹화**의 4단계 구조다. 모든 상황마다 별도 딥러닝 모델을 계속 실행하면 발열·메모리·오탐만 늘어난다.

| 실행 위치 | 최종 구성 | 결정 |
|---|---|---|
| Jetson GPU | **YOLOv8n TensorRT FP16** | 상시 사용 |
| Jetson CPU | **OpenCV MOG2/프레임 차분 + ByteTrack 추적 규칙** | 상시 사용 |
| Jetson GPU | **YOLOv8n-pose TensorRT** | 낙상 후보가 생길 때만 짧게 사용 |
| Jetson CPU | **YAMNet TFLite + RMS/peak 음량** | 상시 사용 |
| Jetson 40-pin GPIO | PIR·문 열림·터치·인증된 연기 센서 | 함께 사용 |
| 제외 | MoveNet, EfficientAT, 시각 fire/smoke 모델, Pi AI HAT+ | 현재 MVP에서 사용하지 않음 |

녹화는 단일 프레임이 아니라 연속 창과 센서 융합으로 확정하고 기존 60초 순환 버퍼를 저장한다.

**현재 최종 추천은 전부 Jetson 배치**다. 카메라·마이크·센서를 한 보드의 monotonic clock으로 기록하므로 구현과 동기화가 가장 단순하다. 다만 Orin Nano에는 NVENC가 없어 H.264는 소프트웨어 인코딩이다. 웹캠이 H.264를 직접 출력하지 않고 Jetson CPU 사용률이나 프레임 드롭이 허용치를 넘을 때만 Pi 5에 GPIO 센서를 분리한다. 녹화 음성을 쓰는 한 마이크와 YAMNet은 Jetson에 유지한다.

## 1. 무엇을 어디에 올리는가: 배치 근거

### 수정된 판단 전제

1. 주 웹캠 영상과 녹화 파일, 녹화에 포함할 마이크 입력은 Jetson이 소유한다.
2. 1차 구현은 PIR·문·터치 센서도 Jetson 40-pin GPIO에 직접 연결한다.
3. Raspberry Pi 5에는 별도 카메라나 AI HAT/NPU가 없다고 가정한다.
4. Jetson Orin Nano는 최대 67 INT8 TOPS와 7~25 W 전력 모드를 제공하지만 **NVENC가 없다**. 따라서 웹캠이 H.264를 직접 출력하지 않으면 libav/libx264 소프트웨어 인코딩이 CPU를 사용한다.

이 조건에서는 데이터가 생기는 곳과 녹화 결정을 한 보드로 모으는 것이 우선이다. Pi 분리는 모델 용량 때문이 아니라 Jetson CPU의 소프트웨어 인코딩 여유, 센서 배선 거리, 장애 격리가 실제로 필요할 때 선택한다.

### 기술별 배치 근거

| 모델·기술 | 입력·계산 특성 | 배치 위치 | 선택 근거 | 다른 위치를 선택할 조건 |
|---|---|---|---|---|
| H.264 스트림·60초 순환 버퍼 | 카메라 원본 프레임과 높은 대역폭 | **Jetson** | 웹캠이 H.264를 출력하면 재인코딩 없이 remux/copy한다. 그렇지 않으면 Orin Nano의 libx264 소프트웨어 인코딩 CPU 부하를 실측한다. | 인코딩 부하가 기준 초과일 때 카메라 포맷·해상도부터 조정 |
| YOLOv8n TensorRT FP16 | 6.25 MiB, 8.7 GFLOPs, GPU 추론에 적합 | **Jetson GPU** | NVIDIA TensorRT 가속을 사용할 수 있고 카메라 프레임이 이미 Jetson에 있다. Pi CPU로 옮기면 영상 전송 또는 낮은 추론률이 필요하다. | Pi에 AI HAT+와 두 번째 카메라를 직접 연결할 때 |
| ByteTrack | YOLO detection box를 시간축으로 연관; 별도 신경망 불필요 | **Jetson CPU** | YOLO 결과를 같은 프로세스에서 바로 소비하므로 프레임·박스 전송과 동기화가 없다. | 탐지 자체를 다른 보드로 옮길 때 |
| OpenCV MOG2·프레임 차분 | 320×180 저해상도 CPU 연산; 모델 파일 없음 | **Jetson CPU** | 카메라 프레임이 로컬에 있고 값싼 1차 움직임 gate로 YOLO/pose 호출 빈도를 줄인다. | Pi가 별도 카메라를 소유할 때 |
| YOLOv8n-pose TensorRT | 6.52 MiB, 다중 사람 keypoint, GPU 부하 추가 | **Jetson GPU, 조건부** | YOLO 사람 ROI와 원본 프레임이 Jetson에 있고 TensorRT 경로를 재사용한다. 상시 실행하지 않아 인코딩·YOLO 자원을 보호한다. | 낙상 기능을 쓰지 않으면 아예 로드하지 않음 |
| MoveNet Lightning TFLite | 4.54 MiB, 192 입력, single-pose CPU 모델 | **현재 미배치** | Pi에서 가볍지만 Jetson의 사람 crop을 계속 전송해야 하고 기본 모델은 한 사람만 처리한다. 모델 속도보다 데이터 경로 복잡성이 더 크다. | 카메라가 Pi에 있거나 단일 사람 crop을 안정적으로 전달할 때 |
| YAMNet TFLite | 3.94 MiB, 16 kHz mono, 약 0.975초 입력, CPU/TFLite | **Jetson CPU** | 녹화 PCM 버퍼를 그대로 읽어 별도 전송·오디오 clock drift가 없다. 약 2 Hz 실행이라 모델 파일 용량도 작다. | Pi에 별도 마이크를 두고 녹화 음성과의 정확한 동기가 불필요할 때 |
| RMS/peak·배경 소음 적응 | 20~50 ms 오디오 프레임의 매우 작은 CPU 연산 | **Jetson CPU** | YAMNet과 같은 PCM 버퍼를 재사용하며 큰 소리의 즉시 gate 역할을 한다. | 마이크 위치가 바뀔 때 함께 이동 |
| PIR·문·터치·연기 센서 | GPIO·I2C/SPI, 작은 이벤트 | **Jetson GPIO** | 센서 edge를 영상과 같은 `CLOCK_MONOTONIC`으로 찍어 네트워크 지연·시계 차이를 제거한다. | 긴 배선이 어렵거나 Jetson GPIO 접근이 곤란할 때 Pi gateway |
| 이벤트 융합·클립 확정 | 작은 timestamp/type/score 메시지 + 순환 버퍼 접근 | **Jetson CPU** | 최종 녹화 파일을 만드는 순환 버퍼가 Jetson에 있으므로 이벤트 시각과 클립 경계를 한곳에서 결정한다. | 별도 로컬 gateway가 버퍼 제어권까지 가질 때 |

### 부하를 나누는 이유

- GPU는 YOLOv8n에 집중시키고 pose는 후보가 생겼을 때만 짧게 실행한다.
- CPU에는 소프트웨어 H.264, YAMNet, MOG2, tracker가 함께 올라갈 수 있으므로 파일 크기보다 동시 실행 부하를 봐야 한다.
- 카메라가 UVC H.264를 지원하면 `v4l2-ctl --list-formats-ext`로 확인하고 encoded stream을 copy/remux해 CPU 인코딩을 피한다.
- 센서까지 Jetson에 두면 최종 녹화 결정과 모든 이벤트 timestamp가 같은 clock에 놓인다.

> 로컬 PC 스모크 테스트는 모델 파일과 입출력 동작을 확인한 결과일 뿐이다. 이 배치의 성능 근거를 확정하려면 Jetson/Pi 실기기에서 동시 실행 FPS·p95·CPU/GPU·온도·드롭 프레임을 측정해야 한다.

## 2. 문 센서와 PIR 실제 구현

### 문 열림: magnetic reed switch

문틀에는 reed switch, 움직이는 문에는 자석을 붙인다. 스위치 한쪽은 Jetson의 **3.3 V-safe GPIO 입력**, 다른 쪽은 GND에 연결하고 pull-up을 사용한다. 긴 배선이나 전압이 불명확한 완제품 모듈은 GPIO에 바로 연결하지 말고 optocoupler/level shifter를 둔다. 5 V를 Jetson GPIO에 직접 넣으면 안 된다. 정확한 물리 pin은 carrier board와 pinmux를 확인한 뒤 정한다.

JetPack 6에서는 deprecated된 GPIO sysfs 대신 `libgpiod` edge event를 사용한다. contact 변화가 50~200 ms 안정적으로 유지될 때만 `DOOR_OPEN` 또는 `DOOR_CLOSED`를 만들고 `time.monotonic_ns()`, sequence, sensor_id를 함께 저장한다. normally-closed loop를 쓰면 선이 끊어진 상태도 fault 후보로 구분하기 쉽다.

실제 녹화 규칙은 `DOOR_OPEN`만으로 침입을 확정하지 않는다. 예를 들어 현관 zone에서 5초 안에 YOLO `person`이 생기면 방문/출입 클립을 확정하고, 사람이 없으면 문 상태 이벤트만 남긴다.

### PIR: 3.3 V-safe digital motion sensor

PIR의 digital OUT을 Jetson GPIO edge input으로 읽는다. 센서 모듈은 공급전압과 OUT 전압이 제품마다 다르므로 데이터시트 또는 멀티미터로 확인하고, 3.3 V-safe가 아니면 level shifter/opto-isolator를 사용한다. 시작 후 안정화 시간, 감도, retrigger 시간도 제품별로 조정한다.

RISING은 움직임 시작, FALLING은 종료 후보로 기록하고 debounce/cooldown을 적용한다. PIR은 열원 변화만 보므로 사람 판정기가 아니다. `PIR_RISE → MOG2 큰 움직임 → YOLO person` 순으로 확인해 녹화를 확정하고, PIR 단독 이벤트는 저전력 wake-up 또는 추론률 상승 신호로 사용한다.

두 센서 모두 영상 프레임을 직접 분석하는 AI가 아니라 **GPIO trigger**다. 같은 Jetson 프로세스의 event bus로 넣고, ring buffer가 이미 가진 프레임 PTS와 연결한다.

## 3. 센서와 영상의 시계 문제

지적한 문제가 맞다. 센서를 Pi에 두면 영상과 센서 이벤트의 clock domain이 달라진다. NTP는 wall clock을 맞출 뿐, 두 장치의 monotonic clock을 같게 만들지 않으며 MQTT 전달 지연과 재전송도 생긴다.

| 배치 | timestamp 방법 | 남는 오차·위험 | 판단 |
|---|---|---|---|
| **전부 Jetson** | 카메라 PTS·오디오 sample counter·GPIO edge를 Jetson monotonic 기준으로 변환 | 캡처 드라이버/큐 지연만 남음 | 가장 단순하고 권장 |
| **센서만 Pi** | Pi가 source monotonic/UTC/sequence를 전송하고 Jetson이 receive monotonic도 기록 | NTP offset + LAN/MQTT 지연, offline replay | 수십 ms 정밀도가 불필요할 때 가능 |
| **마이크까지 Pi** | 위 오차에 독립 audio sample clock drift와 PCM 전달/녹화 동기화 추가 | 긴 클립에서 A/V drift 가능 | 녹화 음성을 쓸 때 비추천 |

Pi 센서 gateway가 꼭 필요하면 `source_utc_ns`, `source_monotonic_ns`, `received_monotonic_ns`, `seq`를 모두 보낸다. 주기적 ping/echo로 RTT와 clock offset을 추정하고, offline 재전송은 receive time이 아니라 보정된 source time으로 정렬한다. 그러나 60초 pre-buffer가 있다는 사실은 이벤트 시각 오차를 없애지 않는다. 클립 시작을 놓칠 가능성만 낮춰 준다.

따라서 현재 한 웹캠 구조에서는 **카메라·녹화 마이크·YAMNet·문·PIR을 Jetson에 모으는 것이 결론**이다.

## 4. 모델·알고리즘 개발 주체

| 모델·기술 | 개발 주체 | 공식 배포·근거 | 라이선스·성격 | 이 프로젝트의 판단 |
|---|---|---|---|---|
| YOLOv8n / YOLOv8n-pose | **Ultralytics** | Ultralytics 문서·GitHub Assets | AGPL-3.0 또는 Enterprise | Jetson 주 영상 모델로 사용 |
| YAMNet | **Google Sound Understanding / TensorFlow 팀** | TensorFlow Hub·Google TFLite | Google AudioSet 기반 범용 음향 모델 | Jetson CPU 주 오디오 모델로 사용 |
| MoveNet Lightning | **Google TensorFlow 팀** | TensorFlow Hub TFLite | 17 keypoint single-pose 모델 | 가볍지만 프레임/crop 전달 복잡성 때문에 현재 미사용 |
| OpenCV MOG2 | **OpenCV 프로젝트**, Zivkovic의 가우시안 혼합 전경 분할 연구 기반 | OpenCV video 모듈 | OpenCV 4.5+ Apache-2.0 | 큰 움직임·카메라 가림에 사용 |
| ByteTrack | **Yifu Zhang 등 연구팀**, FoundationVision/ByteDance에서 공식 구현 공개 | ECCV 2022·FoundationVision GitHub | 별도 딥러닝 모델이 아닌 detection box 연관 추적기 | YOLO 결과 추적에 사용 |
| EfficientAT MN04/MN10 | **Johannes Kepler University Linz, Institute of Computational Perception** 연구팀 | ICASSP 2023·공식 GitHub Releases | MobileNetV3 기반, MIT 저장소 | YAMNet 대비 배포 이득이 부족해 미사용 |
| Raspberry Pi AI HAT+ | **Raspberry Pi + Hailo** | Raspberry Pi 공식 AI stack | 별도 13/26 TOPS NPU 하드웨어 | 두 번째 카메라가 생길 때만 검토 |

큰 소리의 RMS/peak와 프레임 차분은 학습 모델이 아니라 직접 구현하는 신호처리다. 화재·연기 시각 모델은 특정 공식 범용 모델을 선정한 상태가 아니며, 커스텀 데이터 학습이 필요하므로 MVP에서는 제외한다.

## 5. 녹화 가치가 있는 이벤트 후보

| 범주 | 상황 | 권장 검출법 | AI 필요 | 기본 동작 |
|---|---|---|---|---|
| 추억 | 웃음·아기 웃음 | YAMNet 웃음 계열 | 필요 | 자동 녹화 |
| 추억 | 박수·환호·노래 | YAMNet `Applause/Cheering/Singing` | 필요 | 사용자 선택 녹화 |
| 가족 | 아기 울음 | YAMNet `Baby cry` + 지속시간 | 필요 | 녹화 또는 알림 |
| 반려동물 | 개 짖음·낑낑거림 | YAMNet `Dog/Whimper` | 필요 | 사용자 선택 녹화 |
| 활동 | 사람이 들어옴·나감 | YOLOv8n + 추적 + 가상선 | 기존 모델 재사용 | 녹화 |
| 활동 | 큰 움직임·갑작스러운 움직임 | 저해상도 MOG2/프레임 차분/광류 | 불필요 | AI 확인 후 녹화 |
| 안전 | 비명·큰 충격음 | RMS/peak + YAMNet `Screaming/Smash/Crash` | 결합 | 긴급 클립·선택 알림 |
| 안전 | 유리 파손 | YAMNet `Glass/Breaking` + 큰 소리 | 필요 | 긴급 클립 |
| 안전 | 연기·화재 경보기 | 물리 센서 또는 경보기 음향 + YAMNet | 보조만 | 긴급 클립·알림 |
| 안전 | 낙상 후보 | YOLOv8n 사람 박스 + pose + 시간 규칙 | 선택 | 긴급 클립·확인 알림 |
| 방문 | 초인종·노크 | YAMNet `Doorbell/Knock` + 사람 존재 | 결합 | 녹화 |
| 보안 | 카메라 가림·이동·초점 손실 | 밝기·블러·장면 변화 통계 | 불필요 | 장치 이벤트·알림 |
| 보안 | 지정 구역 침입·장시간 머묾 | YOLOv8n + 추적 + zone/time rule | 기존 모델 재사용 | 녹화 |

`얼굴 인식`, 감정 추론, 지속적인 개인별 행동 프로파일링은 MVP에서 제외한다. 저장 가치보다 개인정보·동의·오인식 위험이 크다.

## 6. 큰 소리와 큰 움직임은 별도 AI가 우선이 아니다

### 큰 소리

16 kHz PCM에서 20~50 ms 프레임의 RMS, peak, crest factor와 1초 기준 배경 소음값을 계산한다. 예를 들어 `현재 dBFS > 이동 중앙값 + 15 dB`가 100 ms 이상 유지되면 YAMNet 확인을 즉시 요청한다. 절대 임계값만 쓰면 마이크 거리와 방마다 결과가 달라진다. 웃음처럼 조용할 수 있는 이벤트는 음량 gate와 무관하게 YAMNet을 약 2 Hz로 계속 실행한다.

### 큰 움직임

320×180 grayscale 영상에서 MOG2 또는 프레임 차분으로 전경 면적 비율을 계산한다. 카메라 자체가 움직였는지는 전체 화면 광류/특징점 변환으로 분리한다. 움직임이 크면 YOLOv8n 추론 주기를 잠시 높이고, 사람이 확인되면 녹화를 확정한다. OpenCV MOG2는 가우시안 혼합 기반 전경 분할을 제공한다.

이 방식은 모델 파일이 없고 CPU 비용이 작으며, 딥러닝 모델은 `무슨 소리/누가 움직였는가`를 확인하는 데만 사용한다.

## 7. 모델·알고리즘 후보 비교

| 후보 | 담당 이벤트 | 장점 | 한계 | 추천 보드 | 판정 |
|---|---|---|---|---|---|
| YOLOv8n TensorRT FP16 | 사람·반려동물·일상 물체 | 기존 확정, 다중 객체, 추적 연동 | 행동 자체는 모름 | Jetson | 필수 |
| YAMNet TFLite | 웃음·비명·울음·박수·경보·유리·노크 등 | 521개 범용 음향, 약 3.94 MiB, 로컬 CPU 실측이 빠름 | 한국 가정 임계값 튜닝 필요 | Jetson CPU | 필수 |
| OpenCV MOG2/프레임 차분 | 움직임·카메라 가림 | 모델 없음, 매우 저렴 | 그림자·조명 변화 오탐 | Jetson | 필수 |
| ByteTrack 계열 추적 | 출입·구역·머묾·속도 | 새 AI 모델 없이 YOLO 결과 재사용 | 가림과 재식별 한계 | Jetson | 필수 |
| YOLOv8n-pose TensorRT | 다중 사람 자세·낙상 후보 | YOLO 생태계와 TensorRT export | 약 6.52 MiB, 계속 실행하면 중복 부하 | Jetson | 조건부 |
| MoveNet Lightning TFLite | 단일 사람 17 keypoint | 약 4.54 MiB, 192 입력, 매우 경량 | 기본 모델은 single-pose; 사람 crop 전달 필요 | Pi 5 또는 Jetson CPU | 대안 |
| 시각 fire/smoke 커스텀 YOLO | 불꽃·연기 | 카메라로 조기 후보 감지 가능 | 일반 사전학습에 없음, 데이터·오탐 위험 | Jetson | MVP 보류 |
| EfficientAT MN04 | 범용 음향 challenger | 공개 한국어 샘플 분리 성능 우수 | YAMNet 대비 로컬 CPU 약 4배 지연, PyTorch 배포 | Jetson/Pi | 보류 |

낙상은 pose keypoint 한 장으로 결정하지 않는다. `몸 중심의 급격한 하강 + 박스 가로/세로 비 변화 + 바닥 가까운 자세의 2~3초 지속 + 큰 움직임/충격음`을 함께 본다. 이 기능은 의료·안전 인증 기능으로 표현하지 않고 사용자 확인용 후보 알림으로 제한한다.

In [1]:
from pathlib import Path
import hashlib, platform
import pandas as pd

ROOT = Path.cwd()
if ROOT.name != 'model_benchmark':
    ROOT = ROOT / 'ai' / 'model_benchmark'
MODELS = ROOT / 'models'
names = ['yolov8n.pt', 'yolov8n-pose.pt', 'yamnet.tflite', 'movenet_lightning_f16.tflite', 'efficientat_mn04_as.pt']
rows = []
for name in names:
    p = MODELS / name
    rows.append({'model_file': name, 'exists': p.exists(), 'MiB': round(p.stat().st_size/1024**2, 2) if p.exists() else None, 'sha256': hashlib.sha256(p.read_bytes()).hexdigest() if p.exists() else None})
pd.DataFrame(rows)

,model_file,exists,MiB,sha256
0,yolov8n.pt,True,6.25,f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d...
1,yolov8n-pose.pt,True,6.52,c6fa93dd1ee4a2c18c900a45c1d864a1c6f7aba75d84f9...
2,yamnet.tflite,True,3.94,10c95ea3eb9a7bb4cb8bddf6feb023250381008177ac16...
3,movenet_lightning_f16.tflite,True,4.54,0fac2226112d0371903ca86e3853cec24ef603a0b2f96f...
4,efficientat_mn04_as.pt,True,3.88,899a8c6217063f6941793102a2780b3dd788ab11fce468...


## 8. 다운로드·로컬 스모크 테스트 결과

추가로 `yolov8n-pose.pt`와 `MoveNet Lightning float16 TFLite`를 공식 배포 주소에서 다운로드했다. 동일 PC에서 워밍업 3회 후 20회 측정한 참고 결과다. Jetson/Pi 5 결과로 해석하면 안 된다.

| 모델 | 실행 장치 | 중앙 지연 | p95 | 비고 |
|---|---|---:|---:|---|
| YOLOv8n | RTX 4070 Laptop GPU | 14.25 ms | 18.76 ms | 전체 predict 경로, 640 |
| YOLOv8n-pose | RTX 4070 Laptop GPU | 11.89 ms | 15.61 ms | 640, 4명 keypoint 출력 |
| MoveNet Lightning TFLite | Intel CPU | 2.80 ms | 4.06 ms | 4 thread, 192×192 single-pose |
| YAMNet TFLite | Intel CPU | 약 4~5 ms | 약 8~11 ms | 15,600 samples, 4 thread |

이 측정은 모델 파일과 입출력 파이프라인이 동작한다는 스모크 테스트다. 최종 FPS·전력·온도는 Jetson 전력 모드와 Pi 냉각 조건에서 다시 잰다.

## 9. 보드 조합 비교

### 모델 저장 용량

| 구성 | Jetson 모델 파일 | Pi 모델 파일 | 전체 파일 합계 | 비고 |
|---|---:|---:|---:|---|
| **전부 Jetson, 기본**: YOLOv8n + YAMNet | **10.19 MiB** | 0 | **10.19 MiB** | 최종 기본 |
| **전부 Jetson, pose 포함** | **16.71 MiB** | 0 | **16.71 MiB** | pose는 조건부 load/run |
| 분리, 기본: YOLO on Jetson + YAMNet on Pi | 6.25 MiB | 3.94 MiB | **10.19 MiB** | 전체 용량 절감 없음 |
| 분리, pose 포함 | 12.77 MiB | 3.94 MiB | **16.71 MiB** | 전체 용량 절감 없음 |

수치는 다운로드한 `.pt`/`.tflite` 파일 기준이다. TensorRT engine은 Jetson에서 별도로 생성되므로 파일과 runtime RAM은 더 커진다. 실제 RAM은 카메라 buffer, TensorRT workspace, 입력 해상도와 동시 stream 수에 따라 달라져 파일 크기로 추정하지 않고 실기기에서 측정한다. 즉 **저장 용량은 어느 배치에서도 문제가 아니며, 성능 판단의 핵심은 동시 실행 CPU/GPU와 인코딩 부하**다.

### 성능·운영 비교

| 조합 | Jetson에서 동시에 실행 | Pi 역할 | 예상 병목 | 동기화·복잡도 | 판정 |
|---|---|---|---|---|---|
| **A. 전부 Jetson** | 카메라/녹화, YOLO TRT, MOG2, tracker, YAMNet, RMS, GPIO, 조건부 pose | UI 또는 미사용 | 웹캠이 H.264 미지원이면 libx264 CPU 경합 | 같은 monotonic clock, 네트워크 의존 없음 | **현재 최종 추천** |
| B. 센서만 Pi | A에서 GPIO만 제외 | PIR/문/터치 + MQTT | Jetson 인코딩 부하는 거의 그대로 | 센서 clock offset·MQTT 지연 추가 | 배선이 멀 때 대안 |
| C. 오디오+센서 Pi | YOLO/MOG2/tracker/pose/녹화 | 마이크, YAMNet, RMS, GPIO | Jetson CPU 일부 절감 | 녹화 음성과 A/V clock drift·PCM 경로 복잡 | 녹화 음성을 쓰면 비추천 |
| D. Pi AI HAT+ | 주 카메라 파이프라인 유지 | 별도 카메라의 Hailo YOLO/pose | 카메라가 하나면 기능 중복 | 추가 비용·두 결과 융합 | 두 번째 카메라만 조건부 |

| 성능 항목 | 전부 Jetson | 센서만 Pi | 오디오+센서 Pi | 해석 |
|---|---|---|---|---|
| YOLO/pose GPU 부하 | 동일 | 동일 | 동일 | Pi 분리로 영상 추론은 빨라지지 않음 |
| H.264 CPU 부하 | 동일 | 동일 | 동일 | 카메라 출력 형식과 libx264 여부가 결정 |
| GPIO CPU 부하 | 매우 작음 | Pi로 이동 | Pi로 이동 | 성능 목적으로 분리할 가치가 거의 없음 |
| YAMNet/RMS CPU 부하 | Jetson에 추가 | Jetson에 추가 | Pi로 이동 | 오디오 분리만 일부 CPU를 줄이지만 동기화 비용이 큼 |
| 네트워크·clock 비용 | 없음 | 작은 이벤트와 offset 처리 | PCM/별도 녹음 및 drift 처리 | 동기화는 전부 Jetson이 우세 |
| 현재 실기기 수치 | **미측정** | **미측정** | **미측정** | 아래 동시 부하 시험 전에는 수치 단정 금지 |

전부 Jetson 안에서도 웹캠 출력 형식이 가장 큰 분기다. `v4l2-ctl --list-formats-ext` 결과에 H.264가 있으면 해당 compressed stream을 ring buffer와 송출로 copy/remux하고, AI용 프레임만 decode한다. H.264가 없고 MJPEG/YUYV만 있으면 720p/1080p 각각에서 libx264 동시 부하를 측정한다. Orin Nano에 NVENC가 있다고 가정해서는 안 된다.

현재 PC 스모크 테스트의 지연은 모델 실행 가능성만 확인한다. Jetson의 확정 성능값이 아니므로 FPS를 환산하지 않는다. 실제 보드에서는 30~60분 동안 녹화+송출+YOLO 10~15 FPS+YAMNet 약 2 Hz+MOG2 5~10 FPS+조건부 pose를 동시에 실행해 dropped frames, encode CPU, total CPU/GPU, RAM, 온도, 전력, 각 모델 p95를 기록해야 한다.

## 10. 최종 추천 아키텍처

```text
USB Camera ──> Jetson capture ──> H.264 copy/remux if webcam supports it
                  │              └─ otherwise libx264 software encode
                  ├────────────────> 60s ring buffer / live publisher
                  │
                  ├─ 320x180 MOG2 / tamper detector (5~10 FPS)
                  ├─ YOLOv8n TensorRT FP16 person/pet (10~15 FPS)
                  ├─ tracker + zone / entry / dwell rules
                  └─ suspicious motion only: pose (2~5 FPS, short burst)

USB/Webcam Mic ──> Jetson PCM buffer ──> recording audio + RMS/peak
                  ├─ YAMNet TFLite (~2 inferences/s)
Jetson GPIO ──────┴─ PIR / door reed / touch / certified smoke sensor
                           │
                           └─ local event candidates with monotonic timestamp

Jetson event fusion: candidate window ±2~3s -> confidence/rules -> one event ID
                                      -> save T-60s ~ T+60s clip
```

모든 후보는 Jetson `CLOCK_MONOTONIC` 기준으로 정렬하고 같은 3~5초 창의 후보를 하나의 이벤트로 합친다. 외부 서버에는 UTC를 함께 기록하되, 로컬 클립 경계 계산에는 시스템 시각 변경의 영향을 받지 않는 monotonic timestamp를 사용한다. Pi를 센서 gateway로 분리하는 경우에만 source/receive timestamp와 offset 추정이 추가된다.

## 11. 초기 이벤트 융합 규칙

| 이벤트 | 초기 규칙 예시 | 우선도 |
|---|---|---|
| 웃음 | YAMNet 웃음 이동평균 on-threshold 통과, 2개 창 이상 | 추억 |
| 방문 | Doorbell/Knock + 10초 안에 현관 zone person | 일반 |
| 큰 충격 | 배경 대비 +15 dB + Smash/Crash 상위 점수 | 높음 |
| 유리 파손 | 큰 소리 + Glass/Breaking 2개 창 중 1개 이상 | 높음 |
| 낙상 후보 | 큰 움직임 + 중심 급하강 + 수평 자세 2초 + 선택적으로 충격음 | 높음 |
| 침입 후보 | 지정 시간대 + person이 금지 zone에 2초 이상 | 높음 |
| 카메라 가림 | 밝기 급변/블러/전경 80% 이상이 2초 지속 | 장치 |
| 아기 울음 | Baby cry 점수 3초 이상; cooldown 적용 | 가족 |

임계값 숫자는 출발점일 뿐이다. 이벤트별 positive/hard-negative 현장 데이터로 Precision·Recall·FP/hour를 측정한 뒤 확정한다. 추억 이벤트는 Recall 중심, 긴급 알림은 Precision 중심으로 다르게 조정한다.

## 12. 최종 결정

### 최종적으로 사용할 것

1. **Jetson Orin Nano에 우선 전부 배치**
   - YOLOv8n TensorRT FP16: 사람·반려동물·물체 탐지
   - ByteTrack: 출입·구역 침입·머묾·이동 속도
   - OpenCV MOG2/프레임 차분: 큰 움직임·카메라 가림의 저비용 1차 감지
   - YOLOv8n-pose TensorRT: 낙상 후보가 먼저 감지된 경우에만 2~5 FPS로 짧게 실행
   - YAMNet TFLite + RMS/peak: 녹화 PCM을 재사용해 웃음·비명·울음·충격음 감지
   - PIR·문 reed·터치·인증된 연기 센서: libgpiod edge event
   - 웹캠 H.264 direct stream이면 copy/remux, 아니면 libx264 소프트웨어 인코딩과 60초 순환 버퍼
2. **Raspberry Pi 5**
   - 기본 구성에서는 UI·상태 표시 등 비실시간 역할만 담당하거나 사용하지 않음
   - 센서 배선이 너무 멀 때만 GPIO gateway로 사용하고 MQTT에 source/receive timestamp를 모두 남김
   - 녹화 음성이 필요하면 마이크와 YAMNet을 Pi로 옮기지 않음

### 현재 사용하지 않을 것

- **MoveNet:** 모델은 가볍지만 주 카메라가 Jetson에 있어 Pi로 사람 crop을 전달하는 복잡성이 더 크다.
- **EfficientAT:** YAMNet 대비 공개 샘플 정확도 이득이 작고 PyTorch 배포와 CPU 지연 부담이 크다.
- **시각 fire/smoke 모델:** 검증된 커스텀 데이터가 없고 안전 오탐 위험이 크다. 인증된 센서를 우선한다.
- **Pi AI HAT+:** 현재 한 대의 주 카메라 구조에서는 Jetson과 기능이 중복된다. 두 번째 카메라가 생길 때만 검토한다.

> **한 줄 결론: 현재 한 웹캠 홈캠은 YOLOv8n·YAMNet·MOG2·추적·문/PIR·녹화를 Jetson 한 대에 모으고, pose만 조건부 실행한다. Pi 분리는 실제 Jetson 30~60분 부하 시험 실패나 센서 배선 제약이 확인될 때만 적용한다.**

## 13. 구현·검증 우선순위

1. Jetson에서 `v4l2-ctl --list-formats-ext`로 웹캠 H.264/MJPEG/YUYV 지원과 마이크 장치를 확인
2. H.264 direct stream은 copy/remux하고, 미지원이면 720p·1080p libx264 CPU 부하를 각각 측정
3. Jetson PCM buffer에 RMS/peak + YAMNet label group을 연결
4. YOLOv8n TensorRT + tracker + 저해상도 MOG2를 같은 frame PTS에 연결
5. 문 reed와 PIR을 3.3 V-safe GPIO에 연결하고 libgpiod debounce/edge event 구현
6. 모든 이벤트를 monotonic timestamp로 fusion하고 기존 녹화/`events` 계약에 연결
7. 30~60분 동시 부하 시험으로 dropped frames, CPU/GPU/RAM/온도/전력/p95와 FP/hour 측정
8. 부하 시험이 실패하면 먼저 해상도·추론률·웹캠 포맷을 조정하고, 그 다음 센서 GPIO만 Pi로 분리
9. 낙상이 실제 요구사항일 때만 pose를 추가하고, 화재·연기는 인증 센서를 우선

MVP 권장 이벤트는 `LAUGHTER`, `PERSON_ENTER`, `PERSON_EXIT`, `LARGE_MOTION`, `SCREAM_OR_CRASH`, `DOORBELL_OR_KNOCK`, `CAMERA_TAMPER`다. 아기·반려동물·낙상은 사용자 설정 기능으로 추가한다.

## 14. 공식 출처

- [Ultralytics YOLOv8 공식 문서](https://docs.ultralytics.com/models/yolov8/) — detection/pose 모델과 export 지원
- [TensorFlow MoveNet 공식 튜토리얼](https://www.tensorflow.org/hub/tutorials/movenet) — Lightning/Thunder, 17 keypoints, TFLite 다운로드
- [MediaPipe Pose Landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker) — pose 대안
- [TensorFlow YAMNet 공식 튜토리얼](https://www.tensorflow.org/hub/tutorials/yamnet) — 521 AudioSet 음향 클래스
- [YAMNet 공식 class map](https://github.com/tensorflow/models/blob/master/research/audioset/yamnet/yamnet_class_map.csv) — 비명·울음·박수·초인종·유리·경보기 등 label 확인
- [OpenCV Background Subtraction](https://docs.opencv.org/master/d1/dc5/tutorial_background_subtraction.html) — MOG2/KNN 전경 검출
- [OpenCV License](https://opencv.org/license/) — OpenCV 4.5+ Apache-2.0
- [FoundationVision ByteTrack](https://github.com/FoundationVision/ByteTrack) — ECCV 2022 공식 구현·연구팀
- [JKU EfficientAT 연구 정보](https://research.jku.at/en/publications/efficient-large-scale-audio-tagging-via-transformer-to-cnn-knowle/) — 연구팀·기관·논문
- [Ultralytics License](https://www.ultralytics.com/license) — AGPL-3.0·Enterprise 조건
- [Raspberry Pi AI HAT 공식 문서](https://www.raspberrypi.com/documentation/accessories/ai-hat-plus.html) — 13/26 TOPS NPU
- [Raspberry Pi AI software 공식 문서](https://www.raspberrypi.com/documentation/computers/ai.html) — Hailo YOLOv8와 pose 예제
- [NVIDIA TensorRT 공식 문서](https://docs.nvidia.com/tensorrt/) — Jetson 모델 최적화
- [Jetson Orin Nano Developer Kit 공식 가이드](https://docs.nvidia.com/jetson/orin-nano-devkit/user-guide/latest/index.html) — 최대 67 INT8 TOPS, 7~25 W, JetPack
- [Jetson Orin Nano hardware layout](https://docs.nvidia.com/jetson/orin-nano-devkit/user-guide/latest/hardware_layout.html) — 40-pin expansion header, USB 3.2, CSI connector
- [NVIDIA Orin Nano software encode 문서](https://docs.nvidia.com/jetson/archives/r36.4/DeveloperGuide/SD/Multimedia/SoftwareEncodeInOrinNano.html) — Orin Nano에 NVENC가 없으며 H.264/H.265를 software encode한다는 공식 근거
- [Jetson Orin Nano GPIO·pinmux 가이드](https://docs.nvidia.com/jetson/archives/r36.4.4/DeveloperGuide/HR/JetsonModuleAdaptationAndBringUp/JetsonOrinNxNanoSeries.html) — 40-pin GPIO와 JetPack 6 GPIO 사용 주의

안전 메모: 일반 카메라·오디오 모델의 화재·낙상·침입 판단은 오탐과 미탐이 가능하다. 인증된 화재경보기·문 센서 등을 대체하지 않으며, 긴급 서비스 자동 신고보다 사용자 확인 알림부터 적용한다.